# 05 การตรวจสอบการรั่วไหลของข้อมูล (Data Leakage Audit)

**เป้าหมาย:** ตรวจว่าการแบ่งข้อมูลและรายการ feature ที่อนุญาตจาก Notebook 04 ไม่มีการรั่วไหลในจุดที่ซอฟต์แวร์ตรวจได้ และแยกให้ชัดว่าสิ่งใด **ยังตรวจไม่ได้** (blocker)

**อินพุต:** `data/processed/province_day_panel.csv`, `event_split_assignments.csv` ที่ hash ตรงกับ `split_artifact_manifest.json` และ lineage ย้อนกลับไปยัง `artifact_manifest.json` และ `source_manifest.json`

**การตรวจ (ทำงานบนข้อมูลจริง ไม่ใช่สวิตช์สถานะ):**
1. ตาราง feature availability: ปฏิทิน/จังหวัด อนุญาตภายใต้สมมติฐาน; lag ย้อนหลัง **ถูกบล็อก** จนกว่าจะพิสูจน์ความล่าช้าการรายงาน; ERA5/สภาพอากาศวันเป้าหมาย, การจราจร, จำนวนผู้เสียชีวิต/บาดเจ็บ, ลักษณะอุบัติเหตุ, ID แหล่งข้อมูล และธงตรวจสอบ **ไม่ใช่ feature**
2. ลำดับเวลา (ช่วงวันที่เป้าหมายของแต่ละ split ไม่ซ้อนกัน) และขอบป้ายกำกับ (label window ที่ eligible สิ้นสุดไม่เกินเวลาออกพยากรณ์ครั้งแรกของ split ถัดไป **รวมถึง `diagnostic_2026`** ซึ่งไม่มีแถวใน panel จึงส่ง cutoff ให้โดยตรงจาก config)
3. กลุ่ม source-ID (`event_uid` = ปี + ACC_CODE) ไม่ซ้ำข้าม split
4. ขอบเขตจังหวัดมาจาก Train เท่านั้น
5. allowlist ไม่มี outcome/identifier/ธง และค่าของ feature ไม่ตรงกับเป้าหมายหรือเป้าหมายในอนาคต
6. hash ของอินพุตและ lineage
7. **Negative controls:** ใส่ `deaths`, feature ที่มาจากอนาคต, ช่วงเวลาซ้อนทับ, ป้ายกำกับข้ามขอบ, กลุ่ม ID ซ้ำ, ธงเป็น feature, จังหวัดนอก Train, hash ผิด — แต่ละ control ยืนยันว่าข้อมูลถูกแก้จริง แล้วการตรวจต้อง **raise** และถูกจับได้

**เอาต์พุต:** `reports/data/leakage/leakage_audit.json`, `feature_availability_matrix.csv`, `negative_controls.csv`, `readiness_summary.json`

**ข้อจำกัด:** ผ่านการตรวจ = ผ่าน **เฉพาะการตรวจที่เขียนไว้** ไม่ใช่การยืนยันว่า "ไม่มี leakage" ทั้งหมด; การดูข้อมูลปี 2024–2026 ใน EDA ทำให้ข้อมูลเหล่านี้ไม่ใช่ข้อมูลที่ไม่เคยถูกแตะ

**สัญญาสำหรับขั้นถัดไป:** ขั้นการสร้างโมเดล (ยังไม่อนุญาต) ต้องใช้เฉพาะ `feature_allowlist` และแถว `training_eligible` และต้องแก้ blocker ใน `readiness_summary.json` ก่อน

In [1]:
import os, sys, json, hashlib
from pathlib import Path
import numpy as np, pandas as pd

def find_backend_root():
    env = os.environ.get('ROADAWARE_BACKEND_ROOT')
    if env:
        return Path(env).resolve()
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / 'backend' / 'config' / 'data_pipeline.json').is_file():
            return c / 'backend'
        if c.name == 'backend' and (c / 'config' / 'data_pipeline.json').is_file():
            return c
    raise FileNotFoundError('backend/config/data_pipeline.json not found from ' + str(here) + '; open the repository root or backend/notebooks')

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

BACKEND = find_backend_root()
REPO = BACKEND.parent
CFG = json.loads((BACKEND / 'config' / 'data_pipeline.json').read_text('utf-8'))
OUT = Path(os.environ.get('ROADAWARE_OUTPUT_DIR') or (BACKEND / CFG['output_root'])).resolve()
if os.environ.get('ROADAWARE_REQUIRE_VENV') == '1':
    assert Path(sys.prefix).resolve() == (REPO / '.venv').resolve(), 'kernel is not the project .venv: ' + sys.executable
RAW, INT, PROC = OUT / 'data' / 'raw' / 'accidents', OUT / 'data' / 'interim', OUT / 'data' / 'processed'
LEAK = OUT / 'reports' / 'data' / 'leakage'
LEAK.mkdir(parents=True, exist_ok=True)
man_file, art_file, split_file = RAW / 'source_manifest.json', INT / 'artifact_manifest.json', PROC / 'split_artifact_manifest.json'
for p, prod in ((man_file, '01_data_collection'), (art_file, '03_cleaning_preparation'), (split_file, '04_split_strategy')):
    if not p.is_file():
        raise FileNotFoundError(str(p) + ' missing; run ' + prod + ' first')
MAN, ART, SMAN = (json.loads(p.read_text('utf-8')) for p in (man_file, art_file, split_file))
print('backend:', BACKEND, '| output root:', OUT)

backend: /Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend | output root: /Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend


In [2]:
# FUNCTIONS (notebook 05) - self-contained; tests execute this cell directly
import hashlib
from pathlib import Path
import numpy as np, pandas as pd

TARGET = 'y_recorded_count'

class LeakageAuditError(AssertionError):
    pass

def enforce(label, violations):
    """Raise when an audit returned violations; return True otherwise."""
    if violations:
        raise LeakageAuditError(label + ': ' + ' | '.join(str(v) for v in violations[:5]))
    return True

def audit_hashes(base, entries):
    """Re-hash every listed file under base; returns mismatches/missing files."""
    out = []
    for e in entries:
        p = Path(base) / e['path']
        if not p.is_file():
            out.append('missing ' + e['path'])
        elif hashlib.sha256(p.read_bytes()).hexdigest() != e['sha256']:
            out.append('sha256 mismatch ' + e['path'])
    return out

def audit_feature_names(features, cfg):
    """Explicit allowlist + forbidden names/substrings (outcomes, identifiers, audit flags, lags, weather, traffic)."""
    allow, names = set(cfg['feature_allowlist']), set(cfg['forbidden_feature_names'])
    out = []
    for f in features:
        low = str(f).lower()
        if f in names:
            out.append(f + ': forbidden outcome/identifier/audit name')
        hits = [s for s in cfg['forbidden_feature_substrings'] if s in low]
        if hits:
            out.append(f + ': forbidden substring ' + ','.join(hits))
        if f not in allow:
            out.append(f + ': not in explicit feature_allowlist')
    return out

def audit_feature_values(panel, features, target=TARGET, max_shift=7, tol=0.98):
    """Inspect actual values: a feature equal to (or |r| >= tol with) the target, or the target shifted into the
    future within a province, is rejected regardless of its name."""
    p = panel.sort_values(['province', 'target_date']).reset_index(drop=True)
    refs = {'target_same_day': p[target].astype(float)}
    for k in range(1, max_shift + 1):
        refs['target_future_+%dd' % k] = p.groupby('province')[target].shift(-k).astype(float)
    out = []
    for f in features:
        if f not in p.columns:
            out.append(f + ': missing from panel')
            continue
        x = pd.to_numeric(p[f], errors='coerce').astype(float)
        if x.notna().sum() < 3 or x.nunique() < 2:
            continue  # non-numeric (e.g. province label) or constant: nothing to correlate
        for name, r in refs.items():
            m = x.notna() & r.notna()
            if m.sum() < 3 or r[m].nunique() < 2 or x[m].nunique() < 2:
                continue
            if (x[m] == r[m]).all() or abs(np.corrcoef(x[m], r[m])[0, 1]) >= tol:
                out.append(f + ': values match ' + name)
                break
    return out

def audit_time_order(panel, split_order):
    """Target-date ranges disjoint and chronological in split order; issue time strictly before target start."""
    out = []
    present = [s for s in split_order if s in set(panel['split'])]
    for i, a in enumerate(present):
        da = panel.loc[panel['split'] == a, 'target_date']
        for b in present[i + 1:]:
            db = panel.loc[panel['split'] == b, 'target_date']
            if da.max() >= db.min():
                out.append('%s target dates reach %s, not before first %s date %s' % (a, da.max().date(), b, db.min().date()))
    bad = int((panel['issue_time'] >= panel['target_start']).sum())
    if bad:
        out.append('%d rows issued at/after target start' % bad)
    return out

def audit_label_boundary(panel, split_order, absent_split_first_issue=None):
    """Eligible label windows of split a must end no later than the first issue time of the next split b in
    split_order. If b has no panel rows (e.g. diagnostic_2026) its first issue time must be passed explicitly;
    an unknown cutoff is reported as a violation rather than silently skipped."""
    extra = absent_split_first_issue or {}
    present = set(panel['split'])
    lab = panel[panel['training_eligible'].astype(bool)]
    out = []
    for a, b in zip(split_order[:-1], split_order[1:]):
        if a not in present:
            continue
        cutoff = panel.loc[panel['split'] == b, 'issue_time'].min() if b in present else extra.get(b)
        if cutoff is None or pd.isna(cutoff):
            out.append('no first-issue cutoff known for %s (next split after %s)' % (b, a))
            continue
        n = int((lab.loc[lab['split'] == a, 'target_end'] > cutoff).sum())
        if n:
            out.append('%d eligible %s labels end after first %s issue %s' % (n, a, b, cutoff))
    return out

def audit_group_disjoint(assign, label_assignments):
    """Each source-ID group (event_uid = source_year:ACC_CODE) is one label row in one split; no-ID rows never labels."""
    lab = assign[assign['assignment'].isin(label_assignments)]
    out = []
    n_split = lab.groupby('event_uid')['split'].nunique()
    if (n_split > 1).any():
        out.append('%d source-ID groups appear in more than one split' % int((n_split > 1).sum()))
    dup = int(lab['event_uid'].duplicated(keep=False).sum())
    if dup:
        out.append('%d label rows share a repeated event_uid' % dup)
    if lab['event_uid'].astype(str).str.startswith('diag:').any() or (lab['event_uid'].astype(str).str.strip() == '').any():
        out.append('no-ID diagnostic rows used as labels')
    return out

def audit_scope(panel, assign, label_assignments, train_name='train'):
    """Panel provinces must equal provinces of Train label events (nothing learned from later splits)."""
    tr = set(assign.loc[(assign['split'] == train_name) & assign['assignment'].isin(label_assignments), 'province'])
    pp = set(panel['province'])
    out = []
    if pp - tr:
        out.append('panel provinces not seen in Train: %s' % sorted(pp - tr)[:5])
    if tr - pp:
        out.append('Train provinces missing from panel: %s' % sorted(tr - pp)[:5])
    return out

def audit_flags_not_features(panel, features):
    """Audit columns must not be features and flagged province-days must not be eligible."""
    out = [f + ': audit column used as feature' for f in features
           if f in ('possibly_undercounted', 'training_eligible', 'n_flagged_records', 'flag_reasons', 'flag_scope', 'boundary_excluded', 'label_status')]
    n = int((panel['possibly_undercounted'].astype(bool) & panel['training_eligible'].astype(bool)).sum())
    if n:
        out.append('%d possibly-undercounted province-days marked training_eligible' % n)
    return out

def negative_control(name, fn):
    """fn must alter data, check the alteration happened, then call enforce(...); the control passes only if the audit raises."""
    try:
        fn()
    except LeakageAuditError as e:
        return {'control': name, 'raised': True, 'caught': True, 'message': str(e)[:300]}
    return {'control': name, 'raised': False, 'caught': False, 'message': 'audit did NOT reject the injected leak'}

def feature_availability_matrix(cfg, columns):
    rows = [{'feature': f, 'group': 'static_spatial' if f == 'province' else 'calendar', 'available_at_issue_time': 'yes_under_assumptions',
             'allowed': True, 'reason': 'calendar is known in advance; reported province treated as a fixed unit label (unverified vs coordinates)'}
            for f in cfg['feature_allowlist']]
    rows += [{'feature': '(group) ' + g, 'group': g, 'available_at_issue_time': 'unproven', 'allowed': False, 'reason': why}
             for g, why in cfg['blocked_feature_groups'].items()]
    seen = set(cfg['feature_allowlist'])
    cols = list(columns)
    for c in cols + list(cfg['forbidden_feature_names']):
        if c in seen:
            continue
        seen.add(c)
        v = [s.split(': ', 1)[1] for s in audit_feature_names([c], cfg)]
        rows.append({'feature': c, 'group': 'panel_column' if c in cols else 'source_or_audit_field', 'available_at_issue_time': 'no / not a feature',
                     'allowed': False, 'reason': '; '.join(v)})
    return pd.DataFrame(rows)

In [3]:
prov = {
    'raw_snapshots_vs_source_manifest': audit_hashes(OUT, MAN['files']),
    'interim_vs_artifact_manifest': audit_hashes(OUT, ART['files']),
    'artifact_manifest_built_from_current_source_manifest': [] if ART['input_source_manifest_sha256'] == sha256_file(man_file) else ['source manifest changed since 03'],
    'split_inputs_lineage': audit_hashes(OUT, SMAN['inputs']),
    'split_outputs_vs_split_manifest': audit_hashes(OUT, SMAN['files']),
}
for k, v in prov.items():
    enforce('provenance ' + k, v)
panel = pd.read_csv(PROC / 'province_day_panel.csv', keep_default_na=False, dtype={'province': str, 'flag_reasons': str, 'flag_scope': str})
for c in ('target_date', 'issue_time', 'target_start', 'target_end'):
    panel[c] = pd.to_datetime(panel[c])
assign = pd.read_csv(PROC / 'event_split_assignments.csv', dtype=str, keep_default_na=False)
DIAG = CFG['diagnostic_split']
SPLIT_ORDER = [s['name'] for s in CFG['splits']] + [DIAG['name']]
FC, TZ = CFG['forecast'], CFG['timezone_assumption']
# diagnostic_2026 has no panel rows, so its first issue time (cutoff for the last label split) is passed explicitly
NEXT_CUTOFFS = {DIAG['name']: pd.Timestamp(str(DIAG['year']) + '-01-01', tz=TZ) - pd.Timedelta(days=FC['issue_days_before_target']) + pd.Timedelta(hours=FC['issue_hour_local'])}
LABEL_ASSIGNMENTS = ['label_counted', 'label_counted_boundary_excluded']
FEATURES = list(CFG['feature_allowlist'])
input_hashes = {p.relative_to(OUT).as_posix(): sha256_file(p) for p in (man_file, art_file, split_file)}
print('provenance verified; panel rows:', len(panel), '| assignment rows:', len(assign))
print('explicit cutoff for absent split:', {k: v.isoformat() for k, v in NEXT_CUTOFFS.items()})
pd.Series(input_hashes, name='sha256')

provenance verified; panel rows: 140602 | assignment rows: 127375
explicit cutoff for absent split: {'diagnostic_2026': '2025-12-31T18:00:00+07:00'}


data/raw/accidents/source_manifest.json        708a3607231e4ee2f801f800defcd78ba05fdc60b886ce...
data/interim/artifact_manifest.json            41d0a79ae380dd9ca69671098801ee867b2a0a60665ac9...
data/processed/split_artifact_manifest.json    4be68a534084bc564892810679ff80d7afa97a85d1f88d...
Name: sha256, dtype: object

In [4]:
matrix = feature_availability_matrix(CFG, panel.columns)
matrix.to_csv(LEAK / 'feature_availability_matrix.csv', index=False)
assert set(matrix.loc[matrix['allowed'], 'feature']) == set(FEATURES)
matrix

,feature,group,available_at_issue_time,allowed,reason
0,province,static_spatial,yes_under_assumptions,True,calendar is known in advance; reported provinc...
1,day_of_week,calendar,yes_under_assumptions,True,calendar is known in advance; reported provinc...
2,month,calendar,yes_under_assumptions,True,calendar is known in advance; reported provinc...
3,day_of_month,calendar,yes_under_assumptions,True,calendar is known in advance; reported provinc...
4,day_of_year,calendar,yes_under_assumptions,True,calendar is known in advance; reported provinc...
5,is_weekend,calendar,yes_under_assumptions,True,calendar is known in advance; reported provinc...
6,(group) historical_lags,historical_lags,unproven,False,blocked until reporting delay / availability a...
7,(group) weather_era5_target_day,weather_era5_target_day,unproven,False,blocked; target-day reanalysis is not availabl...
8,(group) traffic,traffic,unproven,False,blocked; no agreed source
9,(group) holidays,holidays,unproven,False,pending an approved Thai holiday calendar source


In [5]:
audits = {
    'feature_names_allowlist_excludes_outcomes_identifiers_flags': lambda p, a, f: audit_feature_names(f, CFG),
    'feature_values_not_target_or_future_target': lambda p, a, f: audit_feature_values(p, f),
    'audit_flags_not_features_and_flagged_days_ineligible': lambda p, a, f: audit_flags_not_features(p, f),
    'time_order': lambda p, a, f: audit_time_order(p, SPLIT_ORDER),
    'label_boundary': lambda p, a, f: audit_label_boundary(p, SPLIT_ORDER, NEXT_CUTOFFS),
    'source_id_groups_disjoint': lambda p, a, f: audit_group_disjoint(a, LABEL_ASSIGNMENTS),
    'province_scope_train_only': lambda p, a, f: audit_scope(p, a, LABEL_ASSIGNMENTS),
}
def run_all(p, a, f):
    for k, fn in audits.items():
        enforce(k, fn(p, a, f))
    return True

results = []
for k, fn in audits.items():
    v = fn(panel, assign, FEATURES)
    results.append({'check': k, 'passed': not v, 'violations': v[:5]})
for k, v in prov.items():
    results.append({'check': 'provenance_' + k, 'passed': not v, 'violations': v[:5]})
results = pd.DataFrame(results)
display(results)
run_all(panel, assign, FEATURES)  # raises LeakageAuditError on any violation in the real artifacts

,check,passed,violations
0,feature_names_allowlist_excludes_outcomes_iden...,True,[]
1,feature_values_not_target_or_future_target,True,[]
2,audit_flags_not_features_and_flagged_days_inel...,True,[]
3,time_order,True,[]
4,label_boundary,True,[]
5,source_id_groups_disjoint,True,[]
6,province_scope_train_only,True,[]
7,provenance_raw_snapshots_vs_source_manifest,True,[]
8,provenance_interim_vs_artifact_manifest,True,[]
9,provenance_artifact_manifest_built_from_curren...,True,[]


True

In [6]:
# Every control first asserts that its alteration really happened (non-vacuous), then runs the audits.
def inject_deaths():
    p = panel.copy()
    p['deaths'] = p[TARGET]  # outcome-derived proxy column
    assert 'deaths' in p.columns
    run_all(p, assign, FEATURES + ['deaths'])

def inject_future_feature():
    p = panel.sort_values(['province', 'target_date']).copy()
    p['smooth_signal'] = p.groupby('province')[TARGET].shift(-1).fillna(0)  # tomorrow's count under an innocent name
    assert p['smooth_signal'].nunique() > 1, 'future feature is constant; control would be vacuous'
    enforce('feature_values_not_target_or_future_target', audit_feature_values(p, FEATURES + ['smooth_signal']))

def inject_temporal_overlap():
    # append a copy of a mid-Train province-day relabelled as Validation -> Validation dates start inside Train
    tr = panel[panel['split'] == 'train']
    row = tr[tr['target_date'] == tr['target_date'].iloc[len(tr) // 2]].head(1).copy()
    row['split'] = 'validation'
    p = pd.concat([panel, row], ignore_index=True)
    assert len(p) == len(panel) + 1 and p.loc[p['split'] == 'validation', 'target_date'].min() < tr['target_date'].max()
    enforce('time_order', audit_time_order(p, SPLIT_ORDER))

def inject_boundary_label():
    # make every Train boundary row (label ends after the first Validation issue) eligible, keeping flags coherent
    p = panel.copy()
    m = (p['split'] == SPLIT_ORDER[0]) & p['boundary_excluded'].astype(bool)
    p.loc[m, ['possibly_undercounted', 'training_eligible']] = [False, True]
    p.loc[m, 'n_flagged_records'] = 0
    assert int(m.sum()) > 0 and int((p['training_eligible'] != panel['training_eligible']).sum()) > 0, 'no row changed; control would be vacuous'
    enforce('label_boundary', audit_label_boundary(p, SPLIT_ORDER, NEXT_CUTOFFS))

def inject_final_boundary_label():
    # last label split vs the absent diagnostic split: uses the explicit cutoff
    p = panel.copy()
    m = (p['split'] == SPLIT_ORDER[-2]) & p['boundary_excluded'].astype(bool)
    p.loc[m, ['possibly_undercounted', 'training_eligible']] = [False, True]
    assert int(m.sum()) > 0
    enforce('label_boundary', audit_label_boundary(p, SPLIT_ORDER, NEXT_CUTOFFS))

def inject_repeated_source_id():
    a = assign.copy()
    row = a[(a['split'] == 'train') & a['assignment'].isin(LABEL_ASSIGNMENTS)].iloc[[0]].copy()
    row['split'], row['event_date'] = 'validation', '2024-06-01'
    a2 = pd.concat([a, row], ignore_index=True)
    assert a2['event_uid'].duplicated().sum() > a['event_uid'].duplicated().sum()
    run_all(panel, a2, FEATURES)

def inject_flag_feature():
    run_all(panel, assign, FEATURES + ['possibly_undercounted'])

def inject_out_of_train_province():
    extra = panel[panel['split'] == 'validation'].head(3).copy()
    extra['province'] = '__seen_only_in_validation__'
    run_all(pd.concat([panel, extra], ignore_index=True), assign, FEATURES)

def inject_hash_mismatch():
    bad = [dict(e, sha256='0' * 64) for e in SMAN['files'][:1]]
    enforce('provenance', audit_hashes(OUT, bad))

controls = pd.DataFrame([negative_control(n, f) for n, f in [
    ('inject_deaths_feature', inject_deaths), ('inject_future_derived_feature', inject_future_feature),
    ('inject_temporal_overlap', inject_temporal_overlap), ('inject_boundary_label_train_to_validation', inject_boundary_label),
    ('inject_boundary_label_last_split_to_diagnostic', inject_final_boundary_label),
    ('inject_repeated_source_id_group', inject_repeated_source_id), ('inject_audit_flag_as_feature', inject_flag_feature),
    ('inject_province_not_in_train', inject_out_of_train_province), ('inject_input_hash_mismatch', inject_hash_mismatch)]])
controls.to_csv(LEAK / 'negative_controls.csv', index=False)
assert controls['caught'].all(), 'a negative control was NOT rejected: ' + ', '.join(controls.loc[~controls['caught'], 'control'])
controls

,control,raised,caught,message
0,inject_deaths_feature,True,True,feature_names_allowlist_excludes_outcomes_iden...
1,inject_future_derived_feature,True,True,feature_values_not_target_or_future_target: sm...
2,inject_temporal_overlap,True,True,time_order: train target dates reach 2023-12-3...
3,inject_boundary_label_train_to_validation,True,True,label_boundary: 77 eligible train labels end a...
4,inject_boundary_label_last_split_to_diagnostic,True,True,label_boundary: 77 eligible reused_test_2025 l...
5,inject_repeated_source_id_group,True,True,source_id_groups_disjoint: 1 source-ID groups ...
6,inject_audit_flag_as_feature,True,True,feature_names_allowlist_excludes_outcomes_iden...
7,inject_province_not_in_train,True,True,province_scope_train_only: panel provinces not...
8,inject_input_hash_mismatch,True,True,provenance: sha256 mismatch data/processed/eve...


In [7]:
blockers = [
    {'blocker': 'publication_availability', 'detail': 'reporting delay / when records become available is unknown, so historical lags are blocked and labels may be incomplete at evaluation time'},
    {'blocker': 'date_ambiguity', 'detail': 'ambiguous D/M vs M/D slash dates are quarantined; their candidate province-days are flagged, not resolved'},
    {'blocker': 'coverage_completeness', 'detail': 'MOT road network only; completeness unknown; zero = no selected recorded event; unlocalized excluded rows may undercount unknown days'},
    {'blocker': 'reused_test', 'detail': '2025 was previously inspected and 2024-2026 were explored in EDA; no untouched independent test set exists'},
    {'blocker': 'route_precision', 'detail': 'province-day is a provisional demo unit; reported province unverified vs coordinates; coordinates only checked against a broad bbox; route-segment unit not agreed'},
    {'blocker': 'no_id_2026', 'detail': '2569/2026 lacks ACC_CODE and is diagnostic only'},
]
readiness = {
    'software_checks_passed': results.loc[results['passed'], 'check'].tolist(),
    'software_checks_failed': results.loc[~results['passed'], 'check'].tolist(),
    'negative_controls_caught': int(controls['caught'].sum()), 'negative_controls_total': int(len(controls)),
    'unresolved_blockers': blockers,
    'final_model_readiness': 'BLOCKED',
    'claim_scope': 'Only the implemented audits above were run on these artifacts. This is NOT a global no-leakage guarantee.',
}
audit = {
    'producer': '05_data_leakage', 'input_sha256': input_hashes,
    'feature_allowlist': FEATURES, 'blocked_feature_groups': CFG['blocked_feature_groups'],
    'explicit_cutoffs_for_absent_splits': {k: v.isoformat() for k, v in NEXT_CUTOFFS.items()},
    'checks': json.loads(results.to_json(orient='records')),
    'negative_controls': json.loads(controls.to_json(orient='records')),
    'data_inspection_log': 'Notebook 02 EDA inspected 2024-2026 descriptively; 2025 was also seen in earlier project work. These years are historical, not untouched.',
    'readiness': readiness,
}
(LEAK / 'leakage_audit.json').write_text(json.dumps(audit, ensure_ascii=False, indent=2, default=str), 'utf-8')
(LEAK / 'readiness_summary.json').write_text(json.dumps(readiness, ensure_ascii=False, indent=2), 'utf-8')
assert results['passed'].all() and controls['caught'].all()
print('implemented software checks passed:', len(readiness['software_checks_passed']), '/', len(results))
print('negative controls rejected:', readiness['negative_controls_caught'], '/', readiness['negative_controls_total'])
print('final_model_readiness:', readiness['final_model_readiness'])
pd.DataFrame(blockers)

implemented software checks passed: 12 / 12
negative controls rejected: 9 / 9
final_model_readiness: BLOCKED


,blocker,detail
0,publication_availability,reporting delay / when records become availabl...
1,date_ambiguity,ambiguous D/M vs M/D slash dates are quarantin...
2,coverage_completeness,MOT road network only; completeness unknown; z...
3,reused_test,2025 was previously inspected and 2024-2026 we...
4,route_precision,province-day is a provisional demo unit; repor...
5,no_id_2026,2569/2026 lacks ACC_CODE and is diagnostic only


## สรุปความพร้อม (ภาษาง่าย)

- **สิ่งที่ตรวจแล้วด้วยซอฟต์แวร์:** ลำดับเวลาของ split, ขอบป้ายกำกับ 18:00 (รวมขอบสุดท้ายก่อน `diagnostic_2026`), กลุ่ม ACC_CODE ไม่ซ้ำข้าม split, ขอบเขตจังหวัดจาก Train, allowlist ไม่มี outcome/ID/ธง, ค่าของ feature ไม่ตรงกับเป้าหมายวันเดียวกันหรืออนาคต, และ hash/lineage ของอินพุต — ผลจริงอยู่ในตารางด้านบนและ `reports/data/leakage/leakage_audit.json`
- **Negative controls:** เมื่อจงใจใส่ `deaths`, feature จากอนาคต, ช่วงเวลาซ้อนทับ, ป้ายกำกับข้ามขอบ, ID ซ้ำ, ธงเป็น feature, จังหวัดนอก Train หรือ hash ผิด — แต่ละ control ยืนยันว่าข้อมูลถูกแก้จริงก่อน แล้วการตรวจต้อง raise ทุกกรณี (ดู `negative_controls.csv`)
- **สิ่งที่ยังไม่ได้แก้ (blocker):** เวลาที่ข้อมูลถูกเผยแพร่/ความล่าช้าการรายงาน, วันที่กำกวม, ความครอบคลุม/ความครบถ้วน, test ปี 2025 ถูกใช้ซ้ำ, ความละเอียดระดับสายทาง และปี 2026 ไม่มี ID
- การผ่านการตรวจ **ไม่ได้** แปลว่า "ไม่มี leakage" โดยรวม — หมายถึงผ่านเฉพาะการตรวจที่เขียนไว้
- **ความพร้อมของโมเดลจริง: BLOCKED** — ยังไม่อนุญาตให้เทรนโมเดล ขั้นต่อไปคือยืนยันแหล่งข้อมูล/วันที่เผยแพร่ ตกลงหน่วยสายทาง และกำหนดชุด test ใหม่ที่ไม่เคยถูกแตะ